<a href="https://colab.research.google.com/github/MLDreamer/AIMathematicallyexplained/blob/main/Rag_retrieval_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %% [markdown]
# # RAG Retrieval Strategies: Every Engine, One Corpus
#
# > *Same 3 documents. Same 3 questions. Same chunk size (220 chars, overlap 40).*
# > *Only the retrieval function changes. Watch what breaks.*
#
# ---
#
# Follow-up to **Chunk Size as an Experimental Variable in RAG Systems** by Sarah Schürch.
# That article showed chunk size changes *which* text comes back.
# This one shows retrieval strategy changes *whether the right text* comes back at all.
#
# | # | Strategy | Year | Core idea |
# |---|---|---|---|
# | 1 | TF-IDF | 1972 | Weighted rare-term overlap |
# | 2 | BM25 | 1994 | Saturated TF + length normalisation |
# | 3 | Dense bi-encoder | 2020 | Neural cosine similarity |
# | 4 | Dense + HNSW | 2016 | Approximate nearest-neighbour index |
# | 5 | Hybrid BM25 + Dense + RRF | 2009/22 | Reciprocal rank fusion |
# | 6 | Dense + Cross-encoder reranking | 2019 | Two-stage precise scoring |
# | 7 | HyDE | 2022 | Hypothetical document embeddings |
#
# **All HuggingFace models. No API keys. Runs on Colab free tier CPU.**
#
# ---
#
# ### The hard pair
# Q2 asks about **green** highlighting. Q3 asks about **yellow** highlighting.
# Semantically near-identical documents. Any strategy that cannot distinguish them
# is giving you the wrong answer — silently, with confidence.

# %% [markdown]
# ## 0 — Install

# %%
# ── takes about 60 seconds on Colab free tier ──────────────────────────────
!pip install -q sentence-transformers rank-bm25 hnswlib transformers scikit-learn

# %% [markdown]
# ## 1 — Corpus, chunking, evaluation harness
#
# **Nothing in this cell changes across experiments.**
# The only thing that changes is the `retrieval_fn` passed to `evaluate()`.

# %%
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline as hf_pipeline
import hnswlib

# ── Corpus ─────────────────────────────────────────────────────────────────
# Three OneLatex documentation paragraphs.
# Same source as the original chunk-size article.

DOCUMENTS = [
    # Doc 0 — the advantage of separating content from formatting
    (
        "OneLatex separates content creation from formatting. "
        "Users write their content in OneNote without worrying about layout. "
        "The formatting is handled automatically when the document is exported. "
        "This separation allows authors to focus on ideas rather than typography or structure. "
        "It also makes collaboration easier because team members can contribute text "
        "without needing to understand LaTeX syntax. "
        "The main advantage is a clean division of responsibility: "
        "content creators do not need to be technical experts."
    ),
    # Doc 1 — green highlighting = configuration setting
    (
        "In OneNote, text highlighted in green is interpreted by OneLatex "
        "as a configuration setting or a system parameter. "
        "These green-highlighted sections are not rendered as regular text in the final document. "
        "Instead, they are parsed and applied as metadata that controls "
        "how the document is structured and exported. "
        "For example, a green-highlighted line might define the document title, "
        "the author name, or a specific export option. "
        "Green highlighting is therefore a signal to the parser, not to the reader."
    ),
    # Doc 2 — yellow highlighting = comment
    (
        "In OneNote, text highlighted in yellow is treated by OneLatex as a comment. "
        "Yellow-highlighted content is visible during editing in OneNote "
        "but is completely excluded from the exported LaTeX document. "
        "This allows authors to leave notes, reminders, or editorial remarks "
        "directly inside the document without those remarks appearing in the final output. "
        "Yellow highlighting is a lightweight annotation mechanism, "
        "similar in spirit to code comments in a programming language."
    ),
]

QUERIES = [
    "Q1: What is the main advantage of separating content creation from formatting in OneLatex?",
    "Q2: How does OneLatex interpret text highlighted in green in OneNote?",
    "Q3: How does OneLatex interpret text highlighted in yellow in OneNote?",
]

GROUND_TRUTH = [0, 1, 2]  # correct document index per query

LABELS = [
    "Doc 0  —  Content vs. formatting (advantage of separation)",
    "Doc 1  —  Green = configuration setting",
    "Doc 2  —  Yellow = comment / annotation",
]

# ── Chunking ───────────────────────────────────────────────────────────────
# chunk_size=220, overlap=40 — identical to original article's middle experiment

def chunk_text(text, chunk_size=220, overlap=40):
    text = text.strip()
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size].strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]

def build_corpus(docs, chunk_size=220, overlap=40):
    """Returns (chunks, chunk_to_doc) — same shape for every experiment."""
    chunks, chunk_to_doc = [], []
    for doc_idx, doc in enumerate(docs):
        for chunk in chunk_text(doc, chunk_size, overlap):
            chunks.append(chunk)
            chunk_to_doc.append(doc_idx)
    return chunks, chunk_to_doc

CHUNKS, CHUNK_TO_DOC = build_corpus(DOCUMENTS)

print(f"Corpus: {len(CHUNKS)} chunks from {len(DOCUMENTS)} documents\n")
for i, c in enumerate(CHUNKS):
    print(f"  Chunk {i}  (doc {CHUNK_TO_DOC[i]})  {repr(c[:72])}...")

# ── Evaluation harness ─────────────────────────────────────────────────────
# retrieval_fn signature:  (query: str, chunks: list[str]) → (top_chunk_idx: int, score: float)

RESULTS = {}  # name → accuracy (0–3)

def evaluate(name, retrieval_fn):
    sep = "=" * 62
    print(f"\n{sep}")
    print(f"  {name}")
    print(sep)
    correct = 0
    for q, gt in zip(QUERIES, GROUND_TRUTH):
        idx, score = retrieval_fn(q, CHUNKS)
        pred_doc   = CHUNK_TO_DOC[idx]
        hit        = pred_doc == gt
        correct   += int(hit)
        mark       = "ok   " if hit else "WRONG"
        print(f"\n  [{mark}]  {q[:55]}")
        print(f"           Retrieved : {LABELS[pred_doc]}")
        print(f"           Expected  : {LABELS[gt]}")
        print(f"           Score     : {score:.5f}")
    print(f"\n  Accuracy: {correct}/{len(QUERIES)}")
    RESULTS[name] = correct
    return correct

# %% [markdown]
# ---
# ## Strategy 1 — TF-IDF  *(1972)*
#
# **Intuition:** Words rare in the corpus but frequent in a document are its signature.
# Relevance = weighted term overlap.
#
# **Math:**
# ```
# TF-IDF(t, d) = [f(t,d) / |d|]  ×  log[ N / df(t) ]
#
# score(q, d)  = TF-IDF_vec(q) · TF-IDF_vec(d)    ← dot product in ℝ|V|
# ```
#
# **Failure mode:** "automobile" and "car" are orthogonal vectors.
# No semantic transfer whatsoever. Green and yellow are both in-vocabulary,
# so the exact-keyword queries Q2/Q3 might still work — but one synonym away from failure.

# %%
def retrieve_tfidf(query, chunks):
    vec          = TfidfVectorizer()
    chunk_matrix = vec.fit_transform(chunks)
    q_vec        = vec.transform([query])
    scores       = sk_cosine(q_vec, chunk_matrix)[0]
    top_idx      = int(scores.argmax())
    return top_idx, float(scores[top_idx])

evaluate("TF-IDF (1972)", retrieve_tfidf)

# %% [markdown]
# ---
# ## Strategy 2 — BM25  *(1994)*
#
# **Intuition:** TF-IDF grew up. Term frequency saturates at k₁+1 (never reaches ∞).
# Long documents are penalised by length normalisation.
#
# **Math:**
# ```
# BM25(q, d) = Σ IDF(t) × f(t,d)·(k₁+1)
#               t∈q        ─────────────────────────────────────────────
#                          f(t,d) + k₁·(1 − b + b·|d|/avgdl)
#
# k₁ = 1.2  (saturation)    b = 0.75  (length normalisation)
#
# As f(t,d) → ∞ :   TF term  →  k₁+1 = 2.2   ← asymptote, not infinity
# ```
#
# **Failure mode:** Same semantic blindness as TF-IDF.
# But green/yellow are exact keyword matches — BM25 handles Q2/Q3 cleanly.

# %%
def retrieve_bm25(query, chunks):
    tokenized = [c.lower().split() for c in chunks]
    bm25      = BM25Okapi(tokenized, k1=1.2, b=0.75)
    scores    = bm25.get_scores(query.lower().split())
    top_idx   = int(scores.argmax())
    return top_idx, float(scores[top_idx])

evaluate("BM25 (1994)", retrieve_bm25)

# %% [markdown]
# ---
# ## Strategy 3 — Dense Bi-encoder  *(2019–2020)*
#
# **Intuition:** Words are not meaning. Project query and document into a shared
# dense vector space. Relevance = cosine similarity.
#
# **Math:**
# ```
# score(q, d) = cos( E_Q(q), E_D(d) )  =  (e_q · e_d) / (‖e_q‖ · ‖e_d‖)
#
# Trained with InfoNCE contrastive loss:
#   ℒ = −log  exp(score(q, d⁺)/τ) / [exp(score(q,d⁺)/τ) + Σⱼ exp(score(q,dⱼ⁻)/τ)]
# ```
#
# **Watch the scores carefully.**
# The original chunk-size article showed that at chunk_size=220,
# the dense cosine scores for Q2 (green) and Q3 (yellow) are dangerously close.
# We reproduce that exact finding below.

# %%
# Load once — MODEL and CHUNK_EMBS are reused by HNSW, Hybrid, Reranking, HyDE
MODEL      = SentenceTransformer("all-MiniLM-L6-v2")
CHUNK_EMBS = MODEL.encode(CHUNKS, normalize_embeddings=True, show_progress_bar=False)

print(f"Chunk embeddings: {CHUNK_EMBS.shape}  (n_chunks × embedding_dim)\n")

# ── Reproduce the green/yellow ambiguity from the original article ─────────
print("Dense cosine scores for Q2 (green):")
q2_emb    = MODEL.encode([QUERIES[1]], normalize_embeddings=True)
q2_scores = (q2_emb @ CHUNK_EMBS.T)[0]
for i, sc in enumerate(q2_scores):
    arrow = " ← TOP" if i == int(q2_scores.argmax()) else ""
    print(f"  Chunk {i} (doc {CHUNK_TO_DOC[i]})  {sc:.5f}  {LABELS[CHUNK_TO_DOC[i]]}{arrow}")

print(f"\nMargin (rank-1 minus rank-2): {sorted(q2_scores)[-1] - sorted(q2_scores)[-2]:.5f}")
print("This is the fragile margin the original article identified.\n")

def retrieve_dense(query, chunks):
    q_emb   = MODEL.encode([query], normalize_embeddings=True)
    scores  = (q_emb @ CHUNK_EMBS.T)[0]
    top_idx = int(scores.argmax())
    return top_idx, float(scores[top_idx])

evaluate("Dense bi-encoder — all-MiniLM-L6-v2 (2020)", retrieve_dense)

# %% [markdown]
# ---
# ## Strategy 4 — Dense + HNSW Index  *(2016)*
#
# **Intuition:** Brute-force cosine search over N vectors is O(N·d).
# At 100M vectors in ℝ¹⁵³⁶ — 154 billion FLOPs per query. HNSW gives O(log N)
# by building a multi-layer navigable graph.
#
# **Math (layer assignment):**
# ```
# l = ⌊ −ln(Uniform(0,1)) · m_L ⌋,    m_L = 1/ln(M)
#
# P(layer ≥ l) = e^(−l/m_L)   →   exponential decay
#
# Layer 0 : all N nodes   (dense, exhaustive neighbourhood)
# Layer k : ~N/eᵏ nodes   (sparse expressway)
# ```
#
# **Note:** On 4 chunks, HNSW and brute-force give identical results.
# The difference appears at 100k+ documents where brute-force becomes impractical.
# M=16, ef_construction=200, ef=50 are production-grade defaults.

# %%
dim      = CHUNK_EMBS.shape[1]    # 384 for all-MiniLM-L6-v2
hnsw_idx = hnswlib.Index(space="cosine", dim=dim)
hnsw_idx.init_index(max_elements=len(CHUNKS), ef_construction=200, M=16)
hnsw_idx.add_items(CHUNK_EMBS, list(range(len(CHUNKS))))
hnsw_idx.set_ef(50)    # query-time recall dial: higher = better recall, slower

print(f"HNSW index built: {hnsw_idx.get_current_count()} nodes, dim={dim}")
print(f"M=16  ef_construction=200  ef_search=50")
print()
print("Complexity comparison (illustrative):")
print(f"  Brute force at N=100M: 100,000,000 × {dim:,} = {100_000_000 * dim:,} FLOPs")
print(f"  HNSW at N=100M (ef=50, M=16): ≈ log₁₆(100M) × 50 × 16 × {dim:,} ≈ {int(7.4 * 50 * 16 * dim):,} FLOPs")
print(f"  Speedup: ~{100_000_000 // (int(7.4 * 50 * 16)):,}×")

def retrieve_hnsw(query, chunks):
    q_emb             = MODEL.encode([query], normalize_embeddings=True)
    labels, distances = hnsw_idx.knn_query(q_emb, k=1)
    # hnswlib cosine space: distance = 1 − similarity
    top_idx = int(labels[0][0])
    score   = float(1.0 - distances[0][0])
    return top_idx, score

evaluate("Dense + HNSW index (2016)", retrieve_hnsw)

# %% [markdown]
# ---
# ## Strategy 5 — Hybrid BM25 + Dense + RRF  *(2009 / 2022)*
#
# **Intuition:** BM25 wins on exact keywords. Dense wins on paraphrases.
# Run both. Fuse with Reciprocal Rank Fusion.
#
# **Math:**
# ```
# RRF(d) = Σᵣ  1 / (k + rankᵣ(d)),    k = 60
#
# Why ranks, not raw scores?
#   BM25 outputs ~14.7   Cosine outputs ~0.873   ← incompatible scales
#   Rank 1 always means "best" regardless of retriever
#
# Why k = 60?
#   rank 1   →  1/61  = 0.0164   (rewarded, not dominant)
#   rank 10  →  1/70  = 0.0143   (close second, still counted)
#   rank 100 →  1/160 = 0.0063   (long tail, minimally counted)
# ```
#
# **This is the closest thing to a free lunch in retrieval engineering.**

# %%
def rrf(ranked_lists, k=60):
    """
    ranked_lists : list of lists, each containing chunk indices (best first)
    Returns      : (sorted_indices, score_dict)
    """
    scores = {}
    for ranked in ranked_lists:
        for rank, chunk_idx in enumerate(ranked):
            scores[chunk_idx] = scores.get(chunk_idx, 0.0) + 1.0 / (k + rank + 1)
    sorted_ids = sorted(scores, key=lambda x: -scores[x])
    return sorted_ids, scores

def retrieve_hybrid_rrf(query, chunks):
    # BM25 ranking
    tokenized    = [c.lower().split() for c in chunks]
    bm25         = BM25Okapi(tokenized, k1=1.2, b=0.75)
    bm25_scores  = bm25.get_scores(query.lower().split())
    bm25_ranked  = list(bm25_scores.argsort()[::-1])        # best first

    # Dense ranking
    q_emb        = MODEL.encode([query], normalize_embeddings=True)
    dense_scores = (q_emb @ CHUNK_EMBS.T)[0]
    dense_ranked = list(dense_scores.argsort()[::-1])       # best first

    # RRF fusion
    fused, score_dict = rrf([bm25_ranked, dense_ranked], k=60)
    top_idx = fused[0]
    return top_idx, float(score_dict[top_idx])

# Show the fusion step for Q2 (the hard one)
print("RRF fusion breakdown for Q2 (green):\n")
tokenized    = [c.lower().split() for c in CHUNKS]
bm25_demo    = BM25Okapi(tokenized, k1=1.2, b=0.75)
bm25_s       = bm25_demo.get_scores(QUERIES[1].lower().split())
bm25_r       = list(bm25_s.argsort()[::-1])

q2_emb_demo  = MODEL.encode([QUERIES[1]], normalize_embeddings=True)
dense_s      = (q2_emb_demo @ CHUNK_EMBS.T)[0]
dense_r      = list(dense_s.argsort()[::-1])

_, rrf_scores = rrf([bm25_r, dense_r])
for i in sorted(rrf_scores, key=lambda x: -rrf_scores[x]):
    bm25_rank  = bm25_r.index(i) + 1
    dense_rank = dense_r.index(i) + 1
    print(f"  Chunk {i} (doc {CHUNK_TO_DOC[i]}):  BM25 rank={bm25_rank}  Dense rank={dense_rank}  RRF={rrf_scores[i]:.5f}")
print()

evaluate("Hybrid BM25+Dense + RRF (2009/2022)", retrieve_hybrid_rrf)

# %% [markdown]
# ---
# ## Strategy 6 — Dense + Cross-encoder Reranking  *(2019)*
#
# **Intuition:** A bi-encoder never lets query and document see each other during encoding.
# A cross-encoder reads the full concatenation — every attention head can attend
# from query tokens to document tokens and back. Slower. Much more precise.
#
# **Math:**
# ```
# s(q, d) = BERT_[CLS]( [CLS] q [SEP] d [SEP] )
#
# Two-stage pipeline:
#   Stage 1 — Dense (fast):    Top-K candidates via cosine similarity
#   Stage 2 — Cross-encoder:   K full BERT forward passes, rerank, return Top-1
#
# Joint attention sees:   "green" in query  ↔  "yellow" in document  →  low score
#                         "green" in query  ↔  "green" in document   →  high score
# ```
#
# **This directly fixes the Q2/Q3 ambiguity the bi-encoder collapses.**
# The model reads both simultaneously. The mismatch is explicit.

# %%
CROSS_ENCODER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_with_reranking(query, chunks, top_k=None):
    # Use all chunks as candidates (on 4 chunks this is trivially cheap)
    # In production: top_k = 50–200 via dense first stage
    if top_k is None:
        candidate_ids = list(range(len(chunks)))
    else:
        q_emb         = MODEL.encode([query], normalize_embeddings=True)
        dense_scores  = (q_emb @ CHUNK_EMBS.T)[0]
        candidate_ids = list(dense_scores.argsort()[::-1][:top_k])

    pairs     = [[query, chunks[i]] for i in candidate_ids]
    ce_scores = CROSS_ENCODER.predict(pairs)

    # Show scores for Q2 (green) to demonstrate the fix
    if "green" in query.lower():
        print("  Cross-encoder scores for Q2 (green):")
        for local_i, global_i in enumerate(candidate_ids):
            doc_i = CHUNK_TO_DOC[global_i]
            print(f"    Chunk {global_i} (doc {doc_i}):  {ce_scores[local_i]:.4f}  {LABELS[doc_i]}")
        print()

    best_local = int(ce_scores.argmax())
    top_idx    = candidate_ids[best_local]
    return top_idx, float(ce_scores[best_local])

evaluate("Dense + Cross-encoder reranking (2019)", retrieve_with_reranking)

# %% [markdown]
# ---
# ## Strategy 7 — HyDE: Hypothetical Document Embeddings  *(2022)*
#
# **Intuition:** Queries and documents live in slightly different embedding regions.
# A short question ≠ a long detailed answer. HyDE generates a hypothetical answer,
# embeds *that*, then retrieves documents similar to the hypothetical.
#
# **Math:**
# ```
# Standard:   d* = argmax  cos( E(q), E(d) )
#                    d
#
# HyDE:       ĥ  = LLM(query)
#             d* = argmax  cos( E(ĥ), E(d) )
#                    d
#
# Why a hallucinated answer helps:
#   E(q) lives in "question-distribution" space
#   E(ĥ) lives in "document-distribution" space
#   Even a factually wrong ĥ is stylistically in the right neighbourhood
#
# Variance reduction (optional):
#   ē = (1/K) · Σₖ E(ĥₖ)   →  average K hypotheticals
# ```
#
# **Model:** google/flan-t5-small  (≈300 MB, runs on Colab free CPU)
# In production: replace with GPT-4, Claude, or any stronger model.

# %%
GENERATOR = hf_pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device=-1,          # -1 = CPU
)

def generate_hypothetical_doc(query):
    prompt = f"Write a detailed passage that directly answers: {query}"
    result = GENERATOR(prompt, max_new_tokens=80, do_sample=False)
    return result[0]["generated_text"]

# Preview what flan-t5-small generates — quality matters here
print("Hypothetical documents from flan-t5-small:\n")
HYPOTHETICALS = {}
for q in QUERIES:
    hyp = generate_hypothetical_doc(q)
    HYPOTHETICALS[q] = hyp
    print(f"Query : {q[:60]}")
    print(f"HyDoc : {hyp[:130]}")
    print()

def retrieve_hyde(query, chunks):
    hyp     = HYPOTHETICALS.get(query) or generate_hypothetical_doc(query)
    hyp_emb = MODEL.encode([hyp], normalize_embeddings=True)
    scores  = (hyp_emb @ CHUNK_EMBS.T)[0]
    top_idx = int(scores.argmax())
    return top_idx, float(scores[top_idx])

evaluate("HyDE — Hypothetical Document Embeddings (2022)", retrieve_hyde)

# %% [markdown]
# ---
# ## Final Scorecard

# %%
print()
print("=" * 65)
print("  FINAL SCORECARD")
print("  Same 3 documents. Same 3 questions. Same chunk size 220/40.")
print("  Only the retrieval function changed.")
print("=" * 65)
print(f"  {'Strategy':<46}  {'Score'}")
print("-" * 65)

for name, score in RESULTS.items():
    bar  = "[" + "█" * score + "·" * (3 - score) + "]"
    note = "  ← hard pair separated" if score == 3 else ("  ← Q2/Q3 ambiguous" if score < 3 else "")
    print(f"  {name:<46}  {score}/3  {bar}{note}")

print("=" * 65)

print("""
Key insight: Q2 (green) and Q3 (yellow) are the hard pair.
Both documents are about OneNote highlighting. Semantically close.
Any strategy that relies only on document-level compression (bi-encoder)
is one vocabulary shift away from silent failure.

What this means at scale:
  3 documents → margins are visible, failures are recoverable
  3 million documents → semantically similar chunks multiply
                        margins shrink → silent failures compound

The hierarchy of reliability:
  BM25              : robust on keywords, blind to paraphrases
  Dense alone       : semantic, fragile on fine distinctions
  Hybrid + RRF      : inherits both strengths, almost always best default
  Cross-encoder     : fixes what bi-encoders collapse, needs two-stage pipeline
  HyDE              : gains on abstract queries, quality bottlenecked by generator

Retrieval errors are silent.
An LLM will turn a wrong chunk into a confident, fluent, wrong answer.
Inspect retrieval in isolation. Trust no single strategy blindly.
""")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Corpus: 9 chunks from 3 documents

  Chunk 0  (doc 0)  'OneLatex separates content creation from formatting. Users write their c'...
  Chunk 1  (doc 0)  's exported. This separation allows authors to focus on ideas rather than'...
  Chunk 2  (doc 0)  't needing to understand LaTeX syntax. The main advantage is a clean divi'...
  Chunk 3  (doc 1)  'In OneNote, text highlighted in green is interpreted by OneLatex as a co'...
  Chunk 4  (doc 1)  'in the final document. Instead, they are parsed and applied as metadata '...
  Chunk 5  (doc 1)  'ne the document title, the author name, or a specific export option. Gre'...
  Chunk 6  (doc 2)  'In OneNote, text highlighted in yellow is treated by OneLatex as a comme'...
  Chunk 7  (doc 2)  'rted LaTeX document. This allows authors to leave notes, reminders, or e'...
  Chunk 8  (doc 2)  'ghlighting is a lightweig

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


  Dense + Cross-encoder reranking (2019)

  [ok   ]  Q1: What is the main advantage of separating content cr
           Retrieved : Doc 0  —  Content vs. formatting (advantage of separation)
           Expected  : Doc 0  —  Content vs. formatting (advantage of separation)
           Score     : 6.01778
  Cross-encoder scores for Q2 (green):
    Chunk 0 (doc 0):  -0.1476  Doc 0  —  Content vs. formatting (advantage of separation)
    Chunk 1 (doc 0):  -10.3552  Doc 0  —  Content vs. formatting (advantage of separation)
    Chunk 2 (doc 0):  -10.6250  Doc 0  —  Content vs. formatting (advantage of separation)
    Chunk 3 (doc 1):  7.4322  Doc 1  —  Green = configuration setting
    Chunk 4 (doc 1):  -5.5691  Doc 1  —  Green = configuration setting
    Chunk 5 (doc 1):  -4.6629  Doc 1  —  Green = configuration setting
    Chunk 6 (doc 2):  4.7064  Doc 2  —  Yellow = comment / annotation
    Chunk 7 (doc 2):  -8.6058  Doc 2  —  Yellow = comment / annotation
    Chunk 8 (doc 2):  -11.0509 

config.json: 0.00B [00:00, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"